# 06_Requested_demo_Broken_report

This notebook shows the report in-line with `launch_report`, then checks report metadata for invalid semantic model references. The visible `Something's wrong with one or more fields` alert is useful for humans, but the automated signal is the metadata check.

In [ ]:
%pip install semantic-link-labs

In [ ]:
import json
import pandas as pd
import sempy_labs as labs
from sempy_labs.report import get_report_json, launch_report
from IPython.display import display

# ── Parameters ────────────────────────────────────────────────────────────────
report_name       = ""    # Power BI report name
report_workspace  = None  # Report workspace (or None for the default workspace)

dataset_name      = ""    # Semantic model name to validate against
dataset_workspace = None  # Semantic model workspace (or None if same as report)
# ──────────────────────────────────────────────────────────────────────────────

# Render the report inline so you can see the current visual state.
launch_report(report=report_name, workspace=report_workspace)

# Build the valid (table, field) set from the target semantic model.
model_objects = labs.list_semantic_model_objects(dataset=dataset_name, workspace=dataset_workspace)
valid_set = set(zip(model_objects["Parent Name"], model_objects["Object Name"]))

# Retrieve the report layout JSON and extract every field reference it contains.
report_json = get_report_json(report=report_name, workspace=report_workspace)


def _find_entity_in_expression(obj):
    """Walk an Expression subtree to locate the SourceRef Entity name."""
    if not isinstance(obj, dict):
        return None
    if "SourceRef" in obj and isinstance(obj["SourceRef"], dict):
        return obj["SourceRef"].get("Entity")
    for v in obj.values():
        result = _find_entity_in_expression(v)
        if result:
            return result
    return None


def _extract_refs(obj, refs=None):
    """
    Recursively collect (table, field) pairs from the report JSON.

    report.json stores page/visual content as double-serialised JSON strings,
    so string values are parsed with json.loads before recursing.
    """
    if refs is None:
        refs = set()

    if isinstance(obj, dict):
        # Pattern 1: {"Entity": "...", "Property": "..."}
        if "Entity" in obj and "Property" in obj:
            refs.add((obj["Entity"], obj["Property"]))
        # Pattern 2: {"Property": "...", "Expression": {... SourceRef ...}}
        elif "Property" in obj and "Expression" in obj:
            entity = _find_entity_in_expression(obj["Expression"])
            if entity:
                refs.add((entity, obj["Property"]))
        # Pattern 3: {"queryRef": "Table.Field"}
        if "queryRef" in obj and isinstance(obj["queryRef"], str) and "." in obj["queryRef"]:
            table, _, field = obj["queryRef"].partition(".")
            refs.add((table, field))
        for v in obj.values():
            _extract_refs(v, refs)

    elif isinstance(obj, list):
        for item in obj:
            _extract_refs(item, refs)

    elif isinstance(obj, str):
        # Double-serialised blobs — parse and recurse.
        if len(obj) > 2 and obj[0] in ('{', '['):
            try:
                _extract_refs(json.loads(obj), refs)
            except (json.JSONDecodeError, ValueError):
                pass

    return refs


all_refs = _extract_refs(report_json)
broken_rows = [
    {"Table Name": entity, "Object Name": prop}
    for entity, prop in sorted(all_refs)
    if (entity, prop) not in valid_set
]
broken_objects = pd.DataFrame(broken_rows)

if broken_objects.empty:
    print("Validation passed — no broken references found.")
else:
    print(f"Validation failed — {len(broken_objects)} broken reference(s) found.")
    display(broken_objects)
    raise ValueError("Broken report detected: one or more semantic model references no longer resolve.")
